In [1]:
import os
import faiss
import kagglehub

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

c:\Users\devas\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data preparation

In [2]:
path = kagglehub.dataset_download("kshitizregmi/jobs-and-job-description")
data = pd.read_csv(os.path.join(path, 'job_title_des.csv'))

model training

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embs = model.encode(data['Job Description'], show_progress_bar=True)

indices = faiss.IndexFlatL2(embs.shape[1])
indices.add(embs)

Batches: 100%|██████████| 72/72 [01:46<00:00,  1.48s/it]


In [4]:
def find_match(state):
    emb = model.encode([state])
    _, idx = indices.search(emb, 1)
    res = data.iloc[idx[0][0]]
    return {
        'Job Title': res['Job Title'],
        'Job Description': res['Job Description'],
    }, res

In [5]:
match = find_match('what skills for developer')
print(match)

({'Job Title': 'Backend Developer', 'Job Description': "Compile and analyze data, processes, and codes to troubleshoot problems and identify areas for improvement.\nCollaborating with the front-end developers and other team members to establish objectives and design more functional, cohesive codes to enhance the user experience.\nDeveloping ideas for new programs, products, or features by monitoring industry developments and trends.\nRecording data and reporting it to proper parties, such as clients or leadership.\nParticipating in continuing education and training to remain current on best practices, learn new programming languages, and better assist other team members.\nTaking lead on projects, as needed.\nBachelor's degree in computer programming, computer science, or a related field.\nMore education or experience may be required.\nFluency or understanding of specific languages, such as Laravel, codeigniter , PHP, or Python, and operating systems may be required.\nStrong understandi

model evaluation

model deployment

In [14]:
save_to = 'mern deployment/prediction'

model.save(f"{save_to}/model.pkl")
np.save(f"{save_to}/embs.npy", embs)
data.to_csv(f"{save_to}/data.csv", index=False)